# Required Assignment 15.2: Applying gradient descent and backpropagation in Python

In [1]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import numpy as np

ModuleNotFoundError: No module named 'tensorflow'

### 1. Optimisation Background

A convex optimization problem is:

\begin{align}\tag{convex opt}
\mathrm{minimize} \  f(x) \quad \mathrm{subject\;to} \  x \in \mathcal{F} \subset \mathbb{R^n}.
\end{align}

A solution $x^\star$ is called the *global minimizer* if for every $x \in \mathcal{F}$ we have
$$ f(x^\star) \leq f(x). $$

A global minimizer exists if 𝐹 is closed and bounded.

The most used algorithm is named the *gradient descent method*. The algorithm first fixes the iteration number $k=0$ and a starting point $x_0 \in \mathbb{R}^n$. Then, the next candidate solution $x_1$ is constructed as $x_{1} = x_0 - \alpha_0 \cdot\nabla f(x_0) $. Here, $\alpha_0 > 0$ is a constant named the *step size*. We can see that $x_1$ is constructed by taking the previous iteration's solution, $x_0$, and going in the $- \nabla f(x_0)$ direction by a step size of $\alpha_0$. The algorithm keeps iterating for $k= 1,2,\ldots$ by the same rule:
$$x_{k+1} = x_{k}-  \alpha_k \cdot \nabla f(x_{k-1}), \tag{gradient descent}$$
and stops when $\nabla{f}(x_k) = 0$.




### Question 1:

Consider the function \begin{align}
\mathrm{minimize} \ f(x) = (x_1 - 2)^2 + (3 \cdot x_2 - 4)^2 \quad \mathrm{subject\;to} \ x \in \mathbb{R}^2.
\end{align} with gradient

 $\nabla f(x) = \begin{bmatrix} 2\cdot x_1 - 4 \\ 18\cdot x_2 - 24  \end{bmatrix}$


 - The function `fn(x1, x2)` that returns the value of the function is provided
 - The gradient function `grad(x1, x2)` that returns a numpy array is provided.


In [ ]:

def fn(x1, x2):
    return (x1 - 2)**2 + (3*x2 - 4)**2
def grad(x1, x2):
    return np.array([2*x1 - 4, 18*x2 - 24])

Using this gradient function, complete the gradient descent loop that:

- Starts at (0,0).
- Applies the gradient descent update iteratively.
- Prints the iterate every 50 steps.
- Stops when the stopping criterion is met.

After convergence, print:

- The number of iterations taken

- The final value of the iterate `xk`






In [ ]:
###GRADED CELL

# Initialization code outside solution block
x0 = np.zeros(2)
alpha = 0.01
iteration = 0
condition = 0
xk = x0.copy()

### BEGIN SOLUTION
while condition == 0: # while gradient is not zero
    iteration = iteration + 1
    if iteration % 50 == 0:
        print(xk)
    xk1 = xk - alpha * grad(xk[0], xk[1])
    if np.linalg.norm(grad(xk1[0], xk1[1])) <= 10 ** -6:
        condition = 1
    xk = xk1
### END SOLUTION

print(f"Converged in {iteration} iterations")
print(f"Final x = {xk}")


In [ ]:
### BEGIN HIDDEN TESTS

# Test 1: Check that grad function returns correct values at a test point
test_point = np.array([2.0, 4/3])
expected_grad = np.array([2*2 - 4, 18*(4/3) - 24])  # should be [0, 0]
student_grad = grad(test_point[0], test_point[1])
assert np.allclose(student_grad, expected_grad, atol=1e-8), \
       "Gradient function grad(x1,x2) returns incorrect value."

# Test 2: Check that final xk is close to analytical minimum
analytical_min = np.array([2.0, 4/3])
assert np.allclose(xk, analytical_min, atol=1e-4), \
       f"Final iterate xk={xk} is not close to analytical minimum {analytical_min}."

# Test 3: Check stopping condition verified
final_grad_norm = np.linalg.norm(grad(xk[0], xk[1]))
assert final_grad_norm <= 1e-6, \
       f"Gradient norm at final xk is {final_grad_norm}, expected <= 1e-6."

# Test 5: Ensure the xk is being updated (not same as initial)
assert not np.allclose(xk, np.zeros(2)), \
       "xk did not update from the initial point."

### END HIDDEN TESTS


In [ ]:
print("Optimal objective value of", round(fn(xk[0], xk[1]),4),\
      "with the solution", np.round(xk,4), "in",  iteration, "iterations.")

### 2. Optimization in Neural Networks

### Question 2:

Find the prediction $\hat{y}$ for the input $x= ( x_1 = 2, \ x_2 = -3)$ of the following neural network with a single hidden layer.
![Drawing](forward.jpeg)


**Answer**
We first compute the outputs of the neurons $r_1$ and $r_2$ on the hidden layer, and then proceed to the output $s$.

- The input to $r_1$ is
$(-1,3,-0.1)\cdot (1,2,-3)
= -1\cdot 1+3\cdot 2+(-0.1)\cdot(-3)
= 5.3$,
so its output is $\max(0,5.3)=5.3$.
- The input to $r_2$ is
$(0.2,-1,0.5)\cdot(1,2,-3) = -3.3$,
so its output is $\max(0,-3.3)=0$.
- The input to $s$ is
$(-0.2,0.4)\cdot(5.3,0) = -1.06$.
Its output, applying the sigmoid function $e^{-1.06}/(1+e^{-1.06})$, is $0.2573$.

If this is a binary classification setting this means that the neural network returns probability
$0.2573$ for class 1 and $0.7427$ for class 0.


**Question** Use `tensorflow` to answer the question given
above

In [ ]:
# 1. Define the input data and target output for demonstration
x_train = np.array([[2, -3]], dtype=np.float32)
y_train = np.array([[1]], dtype=np.float32)  # Example target

# 2. Set up initial weights as shown in your diagram
weights_input_hidden = np.array([
    [3, -1],     # x1 to h1, x1 to h2
    [-0.1, 0.5], # x2 to h1, x2 to h2
    [-1, 0.2]    # bias to h1, bias to h2
], dtype=np.float32)
weights_hidden_output = np.array([
    [-0.2],      # h1 to output
    [0.4]        # h2 to output
], dtype=np.float32)


### Question 2a:

Using the given initial weight values, write TensorFlow code that:

- Defines trainable weight variables for a neural network.
- Uses `tf.Variable` to ensure these weights are updated during training.
- Initializes the weights `W1` and `W2` from the given values.

In [ ]:
###GRADED CELL
W1 = ...
W2 = ...
###BEGIN SOLUTION
# 3. Define trainable weight Variables for TensorFlow
W1 = tf.Variable(weights_input_hidden)
W2 = tf.Variable(weights_hidden_output)
### END SOLUTION
print(W1)
print(W2)

In [ ]:
###BEGIN HIDDEN TESTS
W1_ = tf.Variable(weights_input_hidden)
W2_ = tf.Variable(weights_hidden_output)
assert W1.shape == W1_.shape, "W1 has incorrect shape"
assert W2.shape == W2_.shape, "W2 has incorrect shape"
###END HIDDEN TESTS

### Question 2b:

Write a TensorFlow function `forward_pass` that performs a forward pass through a simple neural network using the following steps:

- Takes input tensor `x`.
- Adds a bias term and store it to `bias`.
- Concatenating a column of ones to `x` and store it to `x_with_bias`.
- Compute the `hidden_linear` by performing a `tf.matmul()` of `x_with_bias`,'W1`.
- Compute the `hidden_relu` on `hidden_linear` by performing `tf.nn.relu()`.
- Computes the hidden layer linear transformation using weight matrix `W1`.
- Applies the ReLU activation function to the hidden layer output.
- Computes the output layer `output_linear` linear transformation using weight matrix `W2`and `hidden_relu` using the `tf.matmul()`
- Applies the sigmoid activation function `tf_sigmoid()` to get the final output`output_sigmoid`.
- Uses the `@tf.function` decorator to optimize performance.



In [ ]:
### GRADED CELL
@tf.function
# 4. Forward pass
def forward_pass(x):
    bias = ...
    x_with_bias = ...
    hidden_linear = ...
    hidden_relu = ...
    output_linear = ...
    output_sigmoid = ...
    return output_sigmoid
### BEGIN SOLUTION
def forward_pass(x):
    bias = tf.ones((tf.shape(x)[0], 1))
    x_with_bias = tf.concat([x, bias], axis=1)
    hidden_linear = tf.matmul(x_with_bias, W1)
    hidden_relu = tf.nn.relu(hidden_linear)
    output_linear = tf.matmul(hidden_relu, W2)
    output_sigmoid = tf.sigmoid(output_linear)
    return output_sigmoid
### END SOLUTION
print(forward_pass(x_train))

In [ ]:
###BEGIN HIDDEN TESTS
def forward_pass_(x):
    bias_ = tf.ones((tf.shape(x)[0], 1))
    x_with_bias_ = tf.concat([x, bias_], axis=1)
    hidden_linear_ = tf.matmul(x_with_bias_, W1)
    hidden_relu_ = tf.nn.relu(hidden_linear_)
    output_linear_ = tf.matmul(hidden_relu_, W2)
    output_sigmoid_ = tf.sigmoid(output_linear_)
    return output_sigmoid_
assert forward_pass(x_train).shape == (1, 1), "forward_pass has incorrect output shape"
assert np.allclose(forward_pass(x_train), forward_pass_(x_train)), "forward_pass output does not match forward_pass_ output"
###END HIDDEN TESTS



### Question 2c:

Using TensorFlow, write code that:

- Defines a mean squared error loss function using `tf.keras.losses.MeanSquaredError()` and store it in `loss_fn`.
- Creates a stochastic gradient descent (SGD) optimizer with a learning rate of 0.1 using `tf.optimizers.SGD()` and store it in `optimizer`.

In [ ]:
### GRADED CELL
# 5. Loss function and Optimizer
loss_fn = ...
optimizer = ...
### BEGIN SOLUTION
loss_fn = tf.keras.losses.MeanSquaredError()
optimizer = tf.optimizers.SGD(learning_rate=0.1)
### END SOLUTION
print(loss_fn)
print(optimizer)


In [ ]:
###BEGIN HIDDEN TESTS
loss_fn_ = tf.keras.losses.MeanSquaredError()
optimizer_ = tf.optimizers.SGD(learning_rate=0.1)
assert loss_fn.name == loss_fn_.name, "loss_fn has incorrect name"
assert optimizer.learning_rate == optimizer_.learning_rate, "optimizer has incorrect learning rate"
###END HIDDEN TESTS

### Question 2d:
Write a TensorFlow function `train_step` that performs a single training iteration with gradient descent. The function should:

- Be decorated with `@tf.function` for performance optimization.
- Take input tensors `x` (features) and `y` (true labels).
- Use `tf.GradientTape()` on `tape` to record operations for automatic differentiation.
- Compute predictions by calling the `forward_pass` function on `x` and store it to `y_pred`.
- Calculate the loss using a pre-defined loss function `loss_fn` and store it to `loss`.
- Compute gradients of the loss with respect to trainable weight variables `W1` and `W2` and store it in `gradients`.
- Apply the gradients using a predefined `optimizer.apply_gradients()` to update `W1` and `W2` using a pre-defined optimizer `optimizer`.
- Return the computed `loss`.



In [ ]:
###GRADED CELL
# 6. Training step with gradient descent
y_pred = ...
loss = ...
gradients = ...
@tf.function
###BEGIN SOLUTION
def train_step(x, y):
    with tf.GradientTape() as tape:
        y_pred = forward_pass(x)
        loss = loss_fn(y, y_pred)
    gradients = tape.gradient(loss, [W1, W2])
    optimizer.apply_gradients(zip(gradients, [W1, W2]))
    return loss
###END SOLUTION
print(train_step(x_train, y_train))



In [ ]:
###BEGIN HIDDEN TESTS

def train_step_(x, y):
    # Reset weights before calculating the expected loss
    W1.assign(weights_input_hidden)
    W2.assign(weights_hidden_output)
    with tf.GradientTape() as tape:
        y_pred_ = forward_pass(x)
        loss_ = loss_fn(y, y_pred_)
    return loss_

assert train_step(x_train, y_train).shape == (), "train_step has incorrect output shape"

# Reset weights before calling train_step for the comparison
W1.assign(weights_input_hidden)
W2.assign(weights_hidden_output)
assert np.allclose(train_step(x_train, y_train), train_step_(x_train, y_train)), "train_step output does not match train_step_ output"
###END HIDDEN TESTS

### Question 2e:

Write a training loop in TensorFlow that:

- Runs for 1000 epochs.
- In each epoch, calls the `train_step` function with training data `x_train` and `y_train`.
- Stores the returned loss from each training step in a variable named `loss`.



In [ ]:
### GRADED CELL
loss = ...
# 7. Training loop (epochs)

### BEGIN SOLUTION
for epoch in range(1000):
    loss = train_step(x_train, y_train)
### END SOLUTION
print(loss)


In [ ]:
###BEGIN HIDDEN TESTS
# for epoch in range(1000):
#     loss_ = train_step(x_train, y_train)
assert loss.shape == (), "train_step has incorrect output shape"
# assert np.allclose(loss, loss_), "train_step output does not match train_step_ output"

###END HIDDEN TESTS

### Question 2f:
Using the trained model, write code to:

- Create a test input tensor `x_test` with the value `[[2, -3]]` and data type `float32`.
- Perform a forward pass through the model using the `forward_pass` function to obtain predictions and store it to `y_pred`.
- Convert the prediction tensor to a NumPy array.
- Print the predicted output value in the format:  
  `Prediction y^ for input x=(2,-3): <predicted_value>`

In [ ]:
###GRADED CELL
x_test = ...
y_pred = ...
# 8. Prediction after training

###BEGIN SOLUTION
 
###END SOLUTION
print(f"Prediction y^ for input x=(2,-3): {y_pred[0][0]}")


In [ ]:
###BEGIN HIDDEN TESTS
x_test_ = np.array([[2, -3]], dtype=np.float32)
y_pred_ = forward_pass(x_test).numpy()
assert x_test.shape == (1, 2), "x_test has incorrect shape"
assert np.allclose(y_pred, y_pred_), "y_pred does not match y_pred_"

###END HIDDEN TESTS

In this assignment, you have built a simple neural network from scratch using TensorFlow, step-by-step, gaining a fundamental understanding of the components involved in training a model with gradient descent.